# AI Agents and Sub-Agents

### Introduction

Sub-agents solve the context bloat problem. When agents use tools with large outputs (web search, file reads, database queries), the context window fills up quickly with intermediate results. Sub-agents isolate this detailed work — the main agent receives only the final result, not the dozens of tool calls that produced it.

**When to use sub-agents:**

✅ Multi-step tasks that would clutter the main agent's context  
✅ Specialized domains that need custom instructions or tools  
✅ Tasks requiring different model capabilities  
✅ When you want to keep the main agent focused on high-level coordination  

**When NOT to use sub-agents:**

❌ Simple, single-step tasks  
❌ When you need to maintain intermediate context  
❌ When the overhead outweighs benefits  

In this notebook we build a single web-search agent first, then evolve it into a **main agent that orchestrates a research sub-agent**.

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model.
- A web search MCP server and (optionally) self-hosted LLMs deployed on Azure Container Apps (see the `infra` folder).

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the self-hosted LLM endpoints, the Foundry endpoint, API key and the MCP server endpoint from the Terraform outputs. You can do this by running the following command.


In [ ]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_qwen_36_35b_a100_fqdn = ! terraform -chdir=infra output -raw aca_qwen_36_35b_a100_fqdn
aca_qwen_36_35b_a100_fqdn = aca_qwen_36_35b_a100_fqdn.n
print("LLM Endpoint:", aca_qwen_36_35b_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


### 1. Enable LangSmith Tracing (optional)

[LangSmith](https://docs.smith.langchain.com/) gives you full visibility into agent runs — every model call, tool call, and sub-agent invocation. Enable tracing so you can inspect how the main agent delegates work to the sub-agent.


In [139]:
import os
import langsmith.utils as lsu

# LangSmith caches env lookups; clear cache when setting vars inside a running notebook
lsu.get_env_var.cache_clear()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"  # ensures LangChain callback tracing is on
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = "project-subagents"  # use an existing project

print("LangSmith tracing enabled:", lsu.tracing_is_enabled())
print("LangSmith endpoint:", os.environ["LANGSMITH_ENDPOINT"])
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

LangSmith tracing enabled: True
LangSmith endpoint: https://eu.api.smith.langchain.com
LangSmith project: project-subagents


### 2. Set Up the LLM Model

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the Azure Foundry OpenAI-compatible endpoint. The commented block shows how the same code can point at an LLM hosted on Azure Container Apps with GPU instead. We then send a basic message to verify the connection.


In [140]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [141]:
response = model.stream([HumanMessage(content="Tell me briefly about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI. I can help with writing, coding, research, brainstorming, explanations, and everyday questions. I don’t have personal experiences or feelings, but I can generate useful responses based on patterns in data I was trained on.

### 3. Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [142]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [143]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks &amp; Pricing",
      "url": "https://onyx.app/llm-leaderboard",
      "description": "<b>The</b> definitive <b>LLM</b> leaderboard \u2014 ranking the best AI <b>models</b> including Claude, GPT, Gemini, DeepSeek, Llama, and more across coding, reasoning, math, agentic, and chat benchmarks. Compare <b>LLM

### 4. Run a Single Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [144]:
import httpx
from langchain.tools import tool
from langsmith import traceable
from markdownify import markdownify

@tool
@traceable
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [145]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_IrWl13ztuzzlVBrA8gT63LJ9)
 Call ID: call_IrWl13ztuzzlVBrA8gT63LJ9
  Args:
    query: latest open LLM models 2026 release parameters context window open-weight 2025 2026
    limit: 50
    searchMode: request
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open LLM models 2026 release parameters context window open-weight 202

I searched 50 result pages and fetched multiple webpages, but one important caveat first:

**There are very few truly new “2026-released” open LLMs with fully stable, official specs across all vendors.**  
So the most reliable answer is a comparison of the **latest prominent open/open-weight model families active in 2026**, prioritizing **official sources** for release date, parameters, and context window.

Also, “open LLM” is used loosely in the ecosystem:
- **Open source**: code/weights and often more artifacts are available under a permissive license.
- **Open weight**: weights are downloadable, but not all training data/code are open.
Many top “open” LLMs in 2026 are really **open-weight**.

---

# Executive summary

As of 2026, the leading open/open-weight LLMs are broadly:

- **Kimi K2.6** — strongest open model tier for coding/agentic workflows, but huge and difficult to self-host.
- **DeepSeek V4 Pro / Flash** — frontier open-weight long-context MoE models with strong reasoning/coding and 1M context.
- **GLM-5.1** — strong agentic coding and long-horizon execution, especially tool use and software tasks.
- **Qwen 3.5 / 3.6 family** — best balance of licensing, capability breadth, and deployability; especially strong for coding and multilingual use.
- **Llama 4 Scout / Maverick** — notable for extreme long context (Scout: 10M) and good multimodal performance, but with Meta’s custom license.
- **Mistral Large 3 / Ministral 3** — strong Apache-licensed open models; Large 3 is capable but less dominant than the very top Chinese MoE families.
- **Gemma 4** — excellent efficiency and clean Apache 2.0 licensing, especially good for local or smaller-scale deployment.
- **DeepSeek R1** — still very important for reasoning, but no longer the newest top general open model in 2026.
- **Phi-4 family** — best among smaller practical open models for local reasoning, but much weaker than frontier MoE giants.
- **OpenAI gpt-oss 20B / 120B** — notable open-weight entrants, but not clearly top overall versus Kimi/DeepSeek/GLM/Qwen in 2026.

---

# Comparison table

## Latest prominent open/open-weight LLMs in 2026

| Model | Developer | Release date | Parameters | Active params | Context window | Main strengths | Main limitations |
|---|---|---:|---:|---:|---:|---|---|
| **Kimi K2.6** | Moonshot AI | 2026 | ~1T total | 32B active | 256K | Excellent coding, agentic workflows, long-horizon tasks | Very large; hard to self-host; modified MIT-style license |
| **DeepSeek V4 Pro** | DeepSeek | Apr 2026 | 1.6T total | 49B active | 1M | Frontier reasoning/coding, long context, strong cost/performance | Enormous infra needs; benchmark claims partly vendor-reported |
| **DeepSeek V4 Flash** | DeepSeek | Apr 2026 | 284B total | 13B active | 1M | Cheaper/faster than Pro, still very strong | Lower ceiling than Pro; still large for local use |
| **GLM-5.1** | Z.AI / Zhipu | 2026 | 744B total | 40B active | 200K | Long-horizon agents, coding, tool use, structured workflows | Huge serving footprint; less common ecosystem than Qwen/Llama |
| **Qwen3.6-27B** | Alibaba/Qwen | Apr 22 2026 | 27.8B | 27.8B | 262K | Strong dense coding model, easier to serve, Apache 2.0 | Below frontier giant MoEs on hardest tasks |
| **Qwen3.6-35B-A3B** | Alibaba/Qwen | Apr 16 2026 | 36B total | 3B active | 262K | Very efficient MoE, good coding/agentic performance | Smaller absolute capability than top giant models |
| **Qwen3.5-397B-A17B** | Alibaba/Qwen | Feb 16 2026 | 397B total | 17B active | 262K | Strong multimodal/coding/agentic model; Apache 2.0 | Still large operationally |
| **Qwen3.5-122B-A10B** | Alibaba/Qwen | Feb 24 2026 | 122B total | 10B active | 262K | Better deployment tradeoff than 397B flagship | Not top-tier frontier |
| **Qwen3.5-27B** | Alibaba/Qwen | Feb 24 2026 | 27B | 27B | 262K | Strong dense open model, easier hosting | Lower ceiling vs giant MoEs |
| **Llama 4 Scout** | Meta | Apr 5 2025, still current in 2026 | 109B total | 17B active | 10M | Best-in-class context length, multimodal, useful for huge corpora | Custom license; real-world 10M use is expensive; not best raw reasoning |
| **Llama 4 Maverick** | Meta | Apr 5 2025, current in 2026 | 400B total | 17B active | 1M | Strong multimodal/chat, better than Scout on many tasks | Custom license; not strongest coding/reasoning vs top 2026 rivals |
| **Mistral Large 3** | Mistral AI | Dec 2025 | 675B total | 41B active | 256K | Apache 2.0, strong multilingual + image understanding | Not generally seen as #1 open model in 2026 |
| **Ministral 3 14B** | Mistral AI | Dec 2025 | 14B | 14B | 256K | Small, multimodal, Apache 2.0, edge-friendly | Much weaker than frontier models |
| **Ministral 3 8B** | Mistral AI | Dec 2025 | 8B | 8B | 256K | Efficient local/small deployment | Limited frontier capability |
| **Ministral 3 3B** | Mistral AI | Dec 2025 | 3B | 3B | 256K | Very lightweight | Far below large models in depth/reasoning |
| **Gemma 4 31B** | Google | 2026 | 30.7B | 30.7B | 256K | Strong dense open model, Apache 2.0, good reasoning/coding for size | Not frontier overall |
| **Gemma 4 26B A4B** | Google | 2026 | 25.2B total | 3.8B active | 256K | Excellent efficiency, local/self-host sweet spot | Not top-tier on hardest tasks |
| **DeepSeek R1** | DeepSeek | Jan 2025, still relevant in 2026 | 671B total | 37B active | 128K | Reasoning-focused, influential, strong math/logic | Older than V4; less balanced for full agentic production |
| **Phi-4** | Microsoft | Jan 2025 | 14B | 14B | 16K | Small reasoning model, MIT license | Very short context; not frontier |
| **Phi-4 Mini** | Microsoft | Jan 2025 | 3.8B | 3.8B | 128K | Small/local deployment | Limited capability ceiling |
| **gpt-oss-120B** | OpenAI | 2025/2026-era open release | 117B total | 5.1B active | 131K | Fast, open-weight, accessible | Usually not ranked above Kimi/DeepSeek/GLM top tier |
| **gpt-oss-20B** | OpenAI | 2025/2026-era open release | 21B total | 3.6B active | 131K | Cheap, fast, usable local/server deployment | Mid-tier capability only |

---

# Best models by capability

## 1. Best overall frontier open models in 2026

### **Kimi K2.6**
**Capabilities**
- Among the strongest open-weight models for:
  - coding
  - long-horizon agentic tasks
  - tool use
  - software engineering
- Frequently ranked at or near the top of open-model leaderboards in 2026.

**Limitations**
- About **1T total params / 32B active**, so it is not realistic for most local users.
- Modified MIT license has branding conditions for very large commercial deployments.
- 256K context is strong, but smaller than DeepSeek V4 or Llama 4 Scout.

### **DeepSeek V4 Pro**
**Capabilities**
- Very strong on reasoning, coding, and long-context work.
- **1M context** makes it strong for large repositories, document collections, and persistent agents.
- Better cost/performance than many proprietary frontier models according to vendor claims and third-party coverage.

**Limitations**
- **1.6T total / 49B active** means very high infra complexity.
- Much of the initial evaluation is still based on vendor benchmarks or leaderboard aggregation.
- Operational complexity is higher than Qwen/Gemma/Phi-sized models.

### **GLM-5.1**
**Capabilities**
- One of the best open models for **agentic engineering** and **long-horizon execution**.
- Strong on coding agents, tool use, structured output, and multi-step workflows.
- Large but efficient by frontier standards.

**Limitations**
- Smaller ecosystem mindshare in Western tooling than Llama/Qwen.
- Still huge for self-hosting.
- 200K context is good, but below V4/Llama 4 Scout extremes.

---

# Best models by use case

## Coding and software engineering
**Top picks**
1. **Kimi K2.6**
2. **GLM-5.1**
3. **DeepSeek V4 Pro**
4. **Qwen3.6-27B / 35B-A3B**
5. **Mistral Devstral/Mistral Large 3 family** if you want Apache and Mistral ecosystem

**Why**
- Kimi, GLM, and DeepSeek are strongest for repo-scale and agentic coding.
- Qwen3.6 is especially attractive because it is newer, easier to deploy, and Apache 2.0 licensed.

**Limitations**
- The very best coding models are enormous.
- Smaller models are cheaper to serve but lose long-horizon robustness and tool-use reliability.

---

## Long-context document analysis
**Top picks**
1. **Llama 4 Scout** — **10M context**
2. **DeepSeek V4 Pro / Flash** — **1M context**
3. **Llama 4 Maverick** — **1M context**
4. **Qwen 3.5 / 3.6 family** — **262K context**
5. **Mistral Large 3 / Ministral 3** — **256K context**

**Why**
- Llama 4 Scout is unmatched on raw input window size.
- DeepSeek V4 gives a more modern frontier reasoning/coding balance with 1M context.
- Qwen/Mistral are easier practical deployment choices.

**Limitations**
- Huge context windows are often more of a systems feature than a guarantee of perfect long-context reasoning.
- Real accuracy at extreme lengths depends on retrieval quality, prompting, and infrastructure.

---

## Local/self-hosted practical deployment
**Top picks**
1. **Gemma 4 26B A4B**
2. **Gemma 4 31B**
3. **Qwen3.6-27B**
4. **Qwen3.6-35B-A3B**
5. **Phi-4 / Phi-4 Mini**
6. **Ministral 3 14B / 8B / 3B**

**Why**
- These offer the best tradeoff between quality and tractability.
- Gemma and Qwen are especially attractive due to licensing and context window.

**Limitations**
- They do not match giant frontier MoEs on hardest reasoning/coding tasks.
- Smaller active-parameter MoEs can be efficient but still operationally tricky.

---

# Model-by-model notes

## Kimi K2.6
- **Release**: 2026
- **Parameters**: ~**1T total**, **32B active**
- **Context**: **256K**
- **Strengths**: top-tier coding, agents, long multi-step execution
- **Weaknesses**: huge; difficult to self-host; modified MIT license

## DeepSeek V4 Pro
- **Release**: **24 Apr 2026**
- **Parameters**: **1.6T total**, **49B active**
- **Context**: **1M**
- **Strengths**: frontier open reasoning/coding, strong long-context
- **Weaknesses**: operationally very heavy; official benchmark claims should be treated cautiously

## DeepSeek V4 Flash
- **Release**: **24 Apr 2026**
- **Parameters**: **284B total**, **13B active**
- **Context**: **1M**
- **Strengths**: much cheaper/faster than Pro; still highly capable
- **Weaknesses**: lower peak capability than Pro

## GLM-5.1
- **Release**: 2026
- **Parameters**: **744B total**, **40B active**
- **Context**: **200K**
- **Strengths**: long-horizon coding/agents/tool use
- **Weaknesses**: giant model; less ubiquitous tooling than Qwen/Llama

## Qwen3.6-27B
- **Release**: **22 Apr 2026**
- **Parameters**: **27.8B**
- **Context**: **262K**
- **Strengths**: dense, practical, strong coding, Apache 2.0
- **Weaknesses**: below top giant MoEs on hardest tasks

## Qwen3.6-35B-A3B
- **Release**: **16 Apr 2026**
- **Parameters**: **36B total**, **3B active**
- **Context**: **262K**
- **Strengths**: highly efficient MoE, strong coding/agentic performance
- **Weaknesses**: not frontier-max capability

## Qwen3.5-397B-A17B
- **Release**: **16 Feb 2026**
- **Parameters**: **397B total**, **17B active**
- **Context**: **262K**
- **Strengths**: very strong multimodal/coding/agentic performance; permissive license
- **Weaknesses**: still large and complex

## Qwen3.5-122B-A10B
- **Release**: **24 Feb 2026**
- **Parameters**: **122B total**, **10B active**
- **Context**: **262K**
- **Strengths**: good compromise between size and power
- **Weaknesses**: not top frontier

## Qwen3.5-27B
- **Release**: **24 Feb 2026**
- **Parameters**: **27B**
- **Context**: **262K**
- **Strengths**: strong dense model, easier serving
- **Weaknesses**: less powerful than large MoEs

## Llama 4 Scout
- **Release**: **5 Apr 2025**
- **Parameters**: **109B total**, **17B active**
- **Context**: **10M**
- **Strengths**: unmatched context, multimodal
- **Weaknesses**: Meta custom license; raw reasoning/coding no longer top of 2026 field

## Llama 4 Maverick
- **Release**: **5 Apr 2025**
- **Parameters**: **400B total** (Meta page says 400B; AA lists ~402B), **17B active**
- **Context**: **1M**
- **Strengths**: good multimodal/chat, large context
- **Weaknesses**: custom license; top competitors surpassed it in many reasoning/coding settings

## Mistral Large 3
- **Release**: **Dec 2025**
- **Parameters**: **675B total**, **41B active**
- **Context**: **256K**
- **Strengths**: Apache 2.0, multilingual, multimodal
- **Weaknesses**: not usually regarded as strongest frontier open model in 2026

## Ministral 3 family
- **Release**: **Dec 2025**
- **Parameters**: **14B**, **8B**, **3B**
- **Context**: **256K**
- **Strengths**: small, multimodal, Apache, edge-friendly
- **Weaknesses**: much weaker than frontier giants

## Gemma 4 31B
- **Release**: 2026
- **Parameters**: **30.7B**
- **Context**: **256K**
- **Strengths**: strong dense open model, clean license
- **Weaknesses**: not best overall frontier

## Gemma 4 26B A4B
- **Release**: 2026
- **Parameters**: **25.2B total**, **3.8B active**
- **Context**: **256K**
- **Strengths**: excellent efficiency/local deployment
- **Weaknesses**: not top raw capability

## DeepSeek R1
- **Release**: **Jan 2025**
- **Parameters**: **671B total**, **37B active**
- **Context**: **128K**
- **Strengths**: reasoning/math landmark model
- **Weaknesses**: older; more specialized than newer all-rounders

## Phi-4
- **Release**: **Jan 2025**
- **Parameters**: **14B**
- **Context**: **16K**
- **Strengths**: small reasoning model, MIT
- **Weaknesses**: short context, weaker overall

---

# Practical ranking in 2026

## If you want the strongest open model overall
- **Kimi K2.6**
- **DeepSeek V4 Pro**
- **GLM-5.1**
- **Qwen3.5-397B-A17B**

## If you want best licensing + strong capability
- **Qwen 3.5 / 3.6 family** — Apache 2.0
- **Mistral 3 family** — Apache 2.0
- **Gemma 4** — Apache 2.0
- **DeepSeek / GLM / Phi** — MIT

## If you want best local/self-hosted tradeoff
- **Gemma 4 26B A4B**
- **Qwen3.6-27B**
- **Qwen3.6-35B-A3B**
- **Phi-4**
- **Ministral 3 14B**

## If you need huge context
- **Llama 4 Scout** — 10M
- **DeepSeek V4 Pro / Flash** — 1M
- **Llama 4 Maverick** — 1M
- **Qwen 3.5 / 3.6** — 262K

---

# Key limitations across open models in 2026

1. **Most frontier open models are too large for normal local deployment.**
   - Kimi, DeepSeek V4 Pro, GLM-5.1 are data-center models.

2. **Benchmarks can overstate real-world reliability.**
   - Agentic coding, tool use, and multi-step planning often degrade outside curated benchmarks.

3. **“Open” rarely means fully open-source in the OSI sense.**
   - Many are open-weight with custom or semi-permissive licenses.

4. **Extreme context windows do not guarantee perfect recall/reasoning.**
   - Long-context quality is still uneven.

5. **Multimodal openness is often weaker than text openness.**
   - Some models expose weights but not all supporting tooling/pipelines equally well.

---

# Bottom line

If your question is **“what are the latest open LLM models in 2026?”**, the most important current families are:

- **Kimi K2.6**
- **DeepSeek V4 Pro / Flash**
- **GLM-5.1**
- **Qwen 3.5 / 3.6**
- **Gemma 4**
- **Mistral 3 family**

If your question is **“which should I actually use?”**:
- **Best frontier performance**: **Kimi K2.6**, **DeepSeek V4 Pro**, **GLM-5.1**
- **Best deployable/open-license balance**: **Qwen 3.6**, **Qwen 3.5**
- **Best local efficiency**: **Gemma 4 26B A4B**
- **Best long-context specialist**: **Llama 4 Scout**

---

# References

1. Meta AI — *The Llama 4 herd: The beginning of a new era of natively multimodal AI innovation* (Apr 5, 2025)  
   https://ai.meta.com/blog/llama-4-multimodal-intelligence/

2. Mistral AI — *Introducing Mistral 3*  
   https://mistral.ai/news/mistral-3

3. Artificial Analysis — *Comparison of Open Source Models*  
   https://artificialanalysis.ai/models/open-source

4. Vellum — *Open Source LLM Leaderboard 2026*  
   https://www.vellum.ai/open-llm-leaderboard

5. ComputingForGeeks — *Open Source LLM Comparison Table (2026)*  
   https://computingforgeeks.com/open-source-llm-comparison/

6. BentoML — *The Best Open-Source LLMs in 2026*  
   https://www.bentoml.com/blog/navigating-the-world-of-open-source-large-language-models

7. IBM — *A list of large language models* (updated Apr 2, 2026)  
   https://www.ibm.com/think/topics/large-language-models-list

8. GitHub QwenLM — *Qwen3.6 repository / release notes*  
   https://github.com/QwenLM/Qwen3.6

9. The Register — *DeepSeek's new models are so efficient...* (Apr 24, 2026)  
   https://www.theregister.com/software/2026/04/24/deepseeks-new-models-offer-big-inference-cost-savings/5227950

10. Hugging Face community article — *Best Open-Source LLM Models in 2026*  
   https://huggingface.co/blog/daya-shankar/open-source-llms

If you want, I can next turn this into:
- a **clean CSV/Markdown table**, or
- a **top 10 only** shortlist, or
- a **“best for local GPU / best for API / best for coding / best for RAG”** decision matrix.

### 5. Working with Sub-Agents

A main agent coordinates subagents as tools. All routing passes through the main agent.

In the subagents architecture, a central main agent (often referred to as a supervisor) coordinates subagents by calling them as tools. The main agent decides which subagent to invoke, what input to provide, and how to combine results. Subagents are stateless—they don’t remember past interactions, with all conversation memory maintained by the main agent. This provides context isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.

![](./images/sub_agents.png)


### When to use

Use the subagents pattern when you have multiple distinct domains (e.g., calendar, email, CRM, database), subagents don’t need to converse directly with users, or you want centralized workflow control. For simpler cases with just a few tools, use a single agent.

### Performance comparison

Different patterns have different performance characteristics. Understanding these tradeoffs helps you choose the right pattern for your latency and cost requirements.
Key metrics:
Model calls: Number of LLM invocations. More calls = higher latency (especially if sequential) and higher per-request API costs.
Tokens processed: Total context window usage across all calls. More tokens = higher processing costs and potential context limits.

In [146]:
from langchain.tools import tool
from langchain.agents import create_agent
from langsmith import traceable

# Create a subagent
subagent = create_agent(model=model, 
                        tools=[fetch_webpage_content],
                        system_prompt=(
""""
You are a URL-focused Research Subagent.
You receive exactly:
one url
one user_query
Available tool:
fetch_webpage_content(url) to retrieve page content.
Task:
Fetch the page content for the provided URL.
Extract only information relevant to user_query.
Produce a concise, faithful summary grounded in that page only.
Rules:
Do not browse other URLs.
Do not add outside knowledge.
If the page is inaccessible or irrelevant, return that clearly.
Prefer concrete facts, definitions, numbers, dates, and claims tied to page text.
"""
# """
#     You are a specialized sub-agent for fetching webpage content. 
#     When given a URL, you fetch the webpage content and return it in markdown format.
# """
))
# subagent = create_agent(model="google_genai:gemini-3.1-pro-preview", tools=[...])

# Wrap it as an async tool so it stays in the async execution path
@tool(description="Fetch a webpage and return markdown content")
@traceable
async def call_sub_agent(query: str):
    result = await subagent.ainvoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# Main agent with subagent as a tool
main_agent = create_agent(model=model, 
                          tools=[search_tool, call_sub_agent],     
                          system_prompt=(
""""
You are the Main Research Orchestrator.
Goal: answer the user query by coordinating web discovery and parallel subagent summarization.
Available tools:
search_tool(query): finds relevant websites.
call_sub_agent(url, user_query): sends one URL and the original query to a subagent and gets back a structured summary.
Required workflow:
Understand the user query and identify intent, scope, and key entities.
Use search_tool to find high-quality candidate URLs.
Select the number of diverse, relevant websites (skip duplicates, low-signal pages, ads, login walls when possible).
If not mentioned, default to 3–6 websites.
For each selected URL, call call_sub_agent(url, user_query).
Merge returned summaries into one synthesized response.
Resolve conflicts by noting disagreement and preferring clearer, better-supported sources.
Rules:
Do not invent facts not present in subagent outputs.
If tools fail, continue with successful results and state limitations.
Keep reasoning concise and focus on user value.
Final response format:
Short answer
Key findings (bullet list)
What sources say (1 bullet per URL)
Gaps / uncertainty
Sources (list of URLs)
"""
# """"
#     You coordinate specialized sub-agent specialized in fetching webpages and formatting to markdown. 
#     Use the task tool to delegate work.
#     Delegate fetch web pages task to the sub agent tool and let it fetch the content of that page and report it back to the main agent.
# """
))


In [147]:
last_message = None

async for step in main_agent.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)


display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_up0FsvST8qkrigjuX4GbiExw)
 Call ID: call_up0FsvST8qkrigjuX4GbiExw
  Args:
    query: latest open LLM models 2026 parameters context window release date open weights model card blog 2026
    limit: 50
    searchMode: auto
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open LLM models 2026 parameters context window release d

Short answer

As of the sources I could successfully fetch, the most clearly documented open/open-weight LLM releases in 2026 are led by **Google Gemma 4** and **Qwen 3.5/3.6**. Among these, **Gemma 4 31B Dense** is presented as the strongest general open model in Google’s lineup, while **Qwen 3.6** is positioned strongly for **agentic coding and repo-level reasoning**. I could not reliably verify some other likely 2026 contenders (for example GLM-5 and the Qwen blog page) because those pages did not fetch usable content, so this answer is a **best-effort comparison from the successfully retrieved sources only**.

Key findings

- I searched the web for **50 result pages** as requested, then fetched a smaller set of the **most relevant primary-source pages**.
- The strongest 2026 open-model evidence I could verify comes from:
  - **Google Gemma 4** blog
  - **Qwen3.6 GitHub**
- Some fetched pages were not usable:
  - `qwen.ai/blog?id=qwen3.6-35b-a3b` returned no meaningful article text
  - `z.ai/blog/glm-5` could not be retrieved
- A Meta page was fetched successfully, but it only documents **Llama 3 (2024)**, so it is not a primary source for “latest open models in 2026.”
- Based on the fetched sources, the latest documented 2026 open/open-weight models include:

| Model | Params | Context window | Release date | Notes |
|---|---:|---:|---|---|
| Gemma 4 E2B | effective 2B | 128K | 2026-04-02 | edge/mobile multimodal |
| Gemma 4 E4B | effective 4B | 128K | 2026-04-02 | edge/mobile multimodal |
| Gemma 4 26B MoE | 26B total, 3.8B active | up to 256K | 2026-04-02 | lower-latency larger model |
| Gemma 4 31B Dense | 31B | up to 256K | 2026-04-02 | best raw quality in Gemma 4 family |
| Qwen3.6-35B-A3B | 35B total, 3B active | 262,144 | 2026-04-16 | agentic coding focus |
| Qwen3.6-27B | 27B | 262,144 | 2026-04-22 | dense coding-oriented model |
| Qwen3.5-397B-A17B | 397B total, 17B active | not stated on fetched page | 2026-02-16 | very large MoE, multilingual/multimodal family claims |
| Qwen3.5-122B-A10B | 122B total, 10B active | not stated on fetched page | 2026-02-24 | open-weight Qwen3.5 |
| Qwen3.5-35B-A3B | 35B total, 3B active | not stated on fetched page | 2026-02-24 | open-weight Qwen3.5 |
| Qwen3.5-27B | 27B | not stated on fetched page | 2026-02-24 | open-weight Qwen3.5 |
| Qwen3.5-9B | 9B | not stated on fetched page | 2026-03-02 | smaller model |
| Qwen3.5-4B | 4B | not stated on fetched page | 2026-03-02 | smaller model |
| Qwen3.5-2B | 2B | not stated on fetched page | 2026-03-02 | smaller model |
| Qwen3.5-0.8B | 0.8B | not stated on fetched page | 2026-03-02 | smallest model |

- For comparison, one successfully fetched older open family page:
  - **Llama 3 8B** — 8B params, 8,192 context, released 2024-04-18
  - **Llama 3 70B** — 70B params, 8,192 context, released 2024-04-18
- Also fetched but older:
  - **Qwen3 (2025)** ranges from **0.6B to 235B-A22B**, with **32K or 128K** context depending on variant, released **2025-04-29**.

What sources say

- **https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/**
  - Documents **Gemma 4** release on **2026-04-02**.
  - Lists four models:
    - **E2B**: effective 2B, 128K
    - **E4B**: effective 4B, 128K
    - **26B MoE**: 26B total / 3.8B active, up to 256K
    - **31B Dense**: 31B, up to 256K
  - Capabilities:
    - advanced reasoning
    - coding
    - multimodal image/video
    - agentic workflows
    - function calling / structured JSON / system instructions
    - native audio input on E2B/E4B
  - Limitations/tradeoffs:
    - edge models optimize for efficiency, not max raw quality
    - 26B MoE optimizes for latency over maximum quality
    - 31B Dense is strongest but heavier to run

- **https://github.com/QwenLM/Qwen3.6**
  - Documents 2026 **Qwen3.5** and **Qwen3.6** releases.
  - Most recent listed:
    - **Qwen3.6-35B-A3B** — 2026-04-16, 35B total / 3B active, 262,144 context
    - **Qwen3.6-27B** — 2026-04-22, 27B, 262,144 context
  - Additional 2026 Qwen3.5 releases:
    - 397B-A17B on 2026-02-16
    - 122B-A10B / 35B-A3B / 27B on 2026-02-24
    - 9B / 4B / 2B / 0.8B on 2026-03-02
  - Capabilities emphasized:
    - agentic coding
    - front-end workflows
    - repository-level reasoning
    - thinking preservation
    - broader Qwen3.5 family claims around multimodal and multilingual support
  - Limitations:
    - fetched page does not state context windows for most Qwen3.5 models
    - lacks explicit model-by-model weakness analysis in text

- **https://qwenlm.github.io/blog/qwen3/**
  - This is a 2025 source, not 2026, but useful as baseline.
  - Lists:
    - dense models from 0.6B to 32B
    - MoE models 30B-A3B and 235B-A22B
  - Context windows:
    - 32K for 0.6B / 1.7B / 4B
    - 128K for larger dense and MoE models
  - Capabilities:
    - hybrid reasoning modes
    - multilingual support across 119 languages/dialects
    - coding and agent improvements
  - Limitation:
    - not a 2026 release

- **https://ai.meta.com/blog/meta-llama-3/**
  - Successfully fetched, but this is a **2024** page.
  - Lists:
    - Llama 3 8B — 8,192 context, 2024-04-18
    - Llama 3 70B — 8,192 context, 2024-04-18
  - Capabilities:
    - improved reasoning, coding, instruction-following, alignment
  - Limitations:
    - text-only at release
    - weaker expected non-English performance than English
    - not current for 2026 latest-open-model comparison

- **https://qwen.ai/blog?id=qwen3.6-35b-a3b**
  - Fetch failed to return useful article text.
  - I could not verify parameter/context/release details from this page directly.

- **https://z.ai/blog/glm-5**
  - Fetch failed.
  - I could not verify GLM-5 details directly from this source.

Gaps / uncertainty

- I did search **50 result pages**, but I did **not fetch all 50 webpages** individually. I fetched the most relevant primary-source pages and some failed.  
- Several likely important 2026 open-model contenders appeared in search results, but I could not safely include them because the fetches were incomplete or unavailable.
- In particular, I could not directly verify:
  - **GLM-5**
  - full **Qwen 3.6 blog article**
  - any newer **Llama 4** primary-source page from the fetched set
- Because of that, this is **not a complete census of every open LLM in 2026**, but a grounded summary of the **latest verifiable ones from successfully fetched sources**.
- For Qwen3.5, the GitHub page gives release dates and model names, but **does not clearly state context windows for every variant**, so those cells remain “not stated on fetched page.”

Sources

- https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/
- https://github.com/QwenLM/Qwen3.6
- https://qwenlm.github.io/blog/qwen3/
- https://ai.meta.com/blog/meta-llama-3/
- https://qwen.ai/blog?id=qwen3.6-35b-a3b
- https://z.ai/blog/glm-5

If you want, I can do a second pass focused only on **primary model-card / GitHub / official blog sources** and produce a cleaner comparison table of the top 10 open models in 2026.

### More Resources

- [LangChain Sub-Agents](https://docs.langchain.com/oss/python/langchain/subagents) — the orchestration pattern used in this notebook
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [LangSmith Tracing](https://docs.smith.langchain.com/) — observe and debug agent runs
- [Open Web Search MCP Server](https://github.com/Aas-ee/open-webSearch) — the web search MCP server used in this notebook
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
